# Project Summary
In this project we analyze the effect of 2 different marketing strategies (the **treatments**) with the aim of maximizing the monetary return by presenting *the right people* with the right strategy.  
The basic idea is that in a group of customers there are 4 types of people: 
1. People who respond positively to being contacted with an offer. These should be the contact phase targets
2. People who buy anyway. Targeting them does not increase the returns.
3. People who would never buy. Targeting them does not increase the returns.
4. People who respond negatively. Targeting them will decrease the returns.   

With this analysis we aim at profiling the customers to figure out who is a type 1 customer, and therefore an optimal target for the campaign. 

# Phase 1: Data Cleaning  

In this notebook we organize the data, we clean it and we create a local SQL database 

In [1]:
%load_ext autoreload
%autoreload 2

import duckdb

from digital_sales.get_data import build_database, download_raw
from digital_sales.paths import DB_PATH

In [2]:
csv_path = download_raw()
build_database(csv_path)

Using existing file: C:\Users\mirko\Documents\Projects\Digital Sales\data\raw\hillstrom.csv
Built C:\Users\mirko\Documents\Projects\Digital Sales\data\warehouse.duckdb


In [3]:
con = duckdb.connect(str(DB_PATH), read_only=True)

con.sql("SHOW TABLES").show()

con.sql("""
    SELECT
        treatment,
        count(*)                         AS customers,
        round(avg(visit) * 100, 2)       AS visit_rate_pct,
        round(avg(conversion) * 100, 2)  AS conversion_rate_pct,
        round(avg(spend), 2)             AS spend_per_customer
    FROM analysis_base
    GROUP BY treatment
    ORDER BY treatment
""").show()

con.close() 

┌─────────────────────┐
│        name         │
│       varchar       │
├─────────────────────┤
│ analysis_base       │
│ campaign_assignment │
│ campaign_outcomes   │
│ customers           │
│ raw_hillstrom       │
└─────────────────────┘

┌──────────────┬───────────┬────────────────┬─────────────────────┬────────────────────┐
│  treatment   │ customers │ visit_rate_pct │ conversion_rate_pct │ spend_per_customer │
│   varchar    │   int64   │     double     │       double        │       double       │
├──────────────┼───────────┼────────────────┼─────────────────────┼────────────────────┤
│ mens_email   │     21307 │          18.28 │                1.25 │               1.42 │
│ no_email     │     21306 │          10.62 │                0.57 │               0.65 │
│ womens_email │     21387 │          15.14 │                0.88 │               1.08 │
└──────────────┴───────────┴────────────────┴─────────────────────┴────────────────────┘



In [4]:
from digital_sales.quality import run_quality_checks

checks = run_quality_checks()
checks

,check_id,category,check_name,severity,n_failing,status
0,1,completeness,"raw file has 64,000 rows",error,0,pass
1,2,completeness,no missing values in the raw file,error,0,pass
2,3,integrity,every customer has a treatment,error,0,pass
3,4,integrity,every customer has an outcome row,error,0,pass
4,5,reconciliation,modelled tables match the raw file row by row,error,0,pass
5,6,reconciliation,treatment labels match the raw segment labels,error,0,pass
6,7,business logic,spend > 0 only for customers who converted,error,0,pass
7,8,business logic,every conversion has spend > 0,error,0,pass
8,9,business logic,every converted also visited,warn,0,pass
9,10,business logic,past-year spend is positive,error,0,pass


In [27]:
con = duckdb.connect(str(DB_PATH), read_only=True)
display(con.sql("SUMMARIZE customers").df())
con.close()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,customer_id,INTEGER,1,64000,62419,32000.5,18475.35295107151,16000,32000,48000,64000,0.0
1,recency,INTEGER,1,12,12,5.763734375,3.5075916120504016,2,5,9,64000,0.0
2,history,"DECIMAL(10,2)",29.99,3345.93,33777,242.08565578125,256.15860829226256,64.80,158.19,326.08,64000,0.0
3,mens,INTEGER,0,1,2,0.55103125,0.49739288006905796,0,1,1,64000,0.0
4,womens,INTEGER,0,1,2,0.54971875,0.4975257918787598,0,1,1,64000,0.0
5,zip_code,VARCHAR,Rural,Urban,3,NaN,NaN,NaN,NaN,NaN,64000,0.0
6,newbie,INTEGER,0,1,2,0.50225,0.49999884373059816,0,1,1,64000,0.0
7,channel,VARCHAR,Multichannel,Web,3,NaN,NaN,NaN,NaN,NaN,64000,0.0
